In [ ]:
import torch.nn as nn
import anndata
import lightning
import torch
import logging
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from typing import List, Dict, Any, Optional
from cellwhisperer.jointemb.dataset.inference import CellxGenePreparationLoader
from cellwhisperer.jointemb.dataset.jointemb import JointEmbedDataModule
from finetuning_eval.models.geneformer import GeneformerCelltypeModel
from cellwhisperer.jointemb.geneformer_model import GeneformerConfig

from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch import Trainer

In [ ]:
# define the LightningModule
class FinetuningModule(lightning.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def step(self, batch, batch_idx):
        # training_step defines the train loop.
        # it is independent of forward
        label = batch.pop("labels")
        pred = self.model(**batch)
        loss = nn.functional.cross_entropy(pred, label)
        return loss
    
    def training_step(self, batch, batch_idx):
        loss = self.step(batch, batch_idx)
        self.log("train_loss", loss)
        return loss
        
    def validation_step(self, batch, batch_idx):
        loss = self.step(batch, batch_idx)
        self.log("val_loss", loss)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [14]:
dm = JointEmbedDataModule(
    tokenizer="bert",  # (corresponds to biobert, but doesn't matter actually)
    transcriptome_processor="geneformer",
    dataset_names="cellxgene_census",
    batch_size=snakemake.params.batch_size,
    train_fraction=0.9,
    include_labels="cell_type"
)

In [ ]:
adata = anndata.read_h5ad(snakemake.input.training_data, backed="r")
num_classes = adata.obs[snakemake.wildcards.label_col].drop_duplicates().shape[0]

model = GeneformerCelltypeModel(GeneformerConfig(), num_classes=num_classes, freeze=snakemake.params.freeze_fm)
finetune_module = FinetuningModule(model)

In [ ]:
training_memory = torch.cuda.memory_allocated()
print(f"Total memory allocated after evaluation: {training_memory}")

Total memory allocated after evaluation: 0


In [ ]:
wandb_logger = WandbLogger(log_model=False)
trainer = lightning.Trainer(max_epochs=snakemake.params.num_epochs, logger=wandb_logger)
trainer.fit(finetune_module, datamodule=dm)

/msc/home/mschae83/miniconda3/envs/cellwhisperer/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /msc/home/mschae83/miniconda3/envs/cellwhisperer/lib ...
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: IPU available: False, using: 0 IPUs
INFO:lightning.pytorch.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/msc/home/mschae83/miniconda3/envs/cellwhisperer/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:72:

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name  | Type                    | Params
--------------------------------------------------
0 | model | GeneformerCelltypeModel | 39.8 M
--------------------------------------------------
247 K     Trainable params
39.6 M    Non-trainable params
39.8 M    Total params
159.354   Total estimated model params size (MB)
INFO:lightning.pytorch.callbacks.model_summary:
  | Name  | Type                    | Params
--------------------------------------------------
0 | model | GeneformerCelltypeModel | 39.8 M
--------------------------------------------------
247 K     Trainable params
39.6 M    Non-trainable params
39.8 M    Total params
159.354   Total estimated model params size (MB)


Training: |                                                                                                   …

In [ ]:
torch.save(model.state_dict(), snakemake.output.model_weights)